In [0]:
%run ./00_setup

# 01 · Discovery
Raio-x das 4 tabelas do sistema de pontos: tamanho, qualidade e como elas se conectam. Serve de base para definir a população, o alvo e as features da EDA.


**Tabelas:** `clientes` · `transacoes` · `transacao_produto` · `produtos`

## 1. Raio-x das tabelas


### 1.1 Clientes
Cadastro dos usuários: chave, nulos, colunas constantes e redes conectadas.

In [0]:
perfil_clientes = perfil_tabela(T_CLIENTES, 'idCliente')

In [0]:
# flags de redes conectadas: quanto da base tem cada uma?
redes = ['flEmail', 'flTwitch', 'flYouTube', 'flBlueSky', 'flInstagram']
pct_redes = spark.table(T_CLIENTES).select([F.avg(c).alias(c) for c in redes]).toPandas().T[0] * 100
fig, ax = plt.subplots()
pct_redes.sort_values().plot.barh(ax=ax, color=COLOR_BLUE)
ax.set(title='% de clientes com cada rede conectada', xlabel='% dos clientes', ylabel='')
salvar(fig, 'T1_clientes_redes')
pct_redes.round(1)

### 1.2 Transações
Cada evento de ganho ou gasto de pontos: período coberto, origem e volume por mês.

In [0]:
perfil_transacoes = perfil_tabela(T_TRANSACOES, 'IdTransacao')

In [0]:
# de onde vêm as transações e como se distribuem os pontos
origem = spark.table(T_TRANSACOES).groupBy('DescSistemaOrigem').count().orderBy(F.desc('count')).toPandas()
origem['%'] = (origem['count'] / origem['count'].sum() * 100).round(2)
display(origem)

sinais = spark.table(T_TRANSACOES).select(
    F.avg((F.col('QtdePontos') > 0).cast('int')).alias('% positivas (ganho)'),
    F.avg((F.col('QtdePontos') < 0).cast('int')).alias('% negativas (gasto)'),
    F.avg((F.col('QtdePontos') == 0).cast('int')).alias('% zero'),
).toPandas().T[0] * 100
sinais.round(2)

In [0]:
# volume ao longo do tempo: período coberto e buracos na série
por_mes = (spark.table(T_TRANSACOES)
           .groupBy(F.date_trunc('month', 'DtCriacao').alias('mes'))
           .agg(F.count('*').alias('transacoes'), F.countDistinct('IdCliente').alias('clientes'))
           .orderBy('mes').toPandas())
fig, ax = plt.subplots()
ax.bar(por_mes['mes'].dt.strftime('%Y-%m'), por_mes['transacoes'], color=COLOR_BLUE)
ax.set(title='Transações por mês', xlabel='mês', ylabel='nº de transações')
ax.tick_params(axis='x', labelrotation=45)
salvar(fig, 'T2_transacoes_por_mes')
por_mes

### 1.3 Transação × Produto
Itens de cada transação: qual produto gerou os pontos.

In [0]:
perfil_transacao_produto = perfil_tabela(T_TRANSACAO_PRODUTO, 'idTransacaoProduto')

### 1.4 Produtos
Catálogo de produtos e categorias.

In [0]:
perfil_produtos = perfil_tabela(T_PRODUTOS, 'IdProduto')

In [0]:
# catálogo completo: poucos produtos, vale ver todos
display(spark.table(T_PRODUTOS).orderBy('DescCategoriaProduto', 'DescNomeProduto').toPandas())

### 1.5 Resumo das tabelas
Visão consolidada de linhas, colunas e chaves.

In [0]:
resumo_tabelas = pd.DataFrame({
    'tabela': ['clientes', 'transacoes', 'transacao_produto', 'produtos'],
    'linhas': [spark.table(t).count() for t in (T_CLIENTES, T_TRANSACOES, T_TRANSACAO_PRODUTO, T_PRODUTOS)],
    'colunas': [len(spark.table(t).columns) for t in (T_CLIENTES, T_TRANSACOES, T_TRANSACAO_PRODUTO, T_PRODUTOS)],
    'colunas com nulo': [(p['nulos'] > 0).sum() for p in (perfil_clientes, perfil_transacoes,
                                                          perfil_transacao_produto, perfil_produtos)],
    'chave': ['idCliente', 'IdTransacao', 'idTransacaoProduto', 'IdProduto'],
})
resumo_tabelas.to_csv(f'{TABLE_DIR}/resumo_tabelas.csv', index=False)
resumo_tabelas

## 2. Relacionamento entre as tabelas
Cardinalidade e integridade de cada join, para garantir que as agregações da EDA não dupliquem linhas.

### 2.1 Clientes × Transações
Quantos clientes transacionam e quantas transações cada um tem (1:N).

In [0]:
spark.sql(f"""
    SELECT
        (SELECT COUNT(DISTINCT IdCliente) FROM {T_TRANSACOES})                         AS clientes_com_transacao,
        (SELECT COUNT(*) FROM {T_CLIENTES})                                            AS clientes_cadastrados,
        (SELECT COUNT(DISTINCT t.IdCliente) FROM {T_TRANSACOES} t
            LEFT JOIN {T_CLIENTES} c ON t.IdCliente = c.idCliente
            WHERE c.idCliente IS NULL)                                                 AS clientes_em_transacoes_sem_cadastro,
        (SELECT COUNT(*) FROM {T_CLIENTES} c
            LEFT JOIN (SELECT DISTINCT IdCliente FROM {T_TRANSACOES}) t ON c.idCliente = t.IdCliente
            WHERE t.IdCliente IS NULL)                                                 AS clientes_sem_nenhuma_transacao
""").toPandas().T.rename(columns={0: 'qtd'})

In [0]:
# transações por cliente: a relação é 1 cliente -> N transações
tx_cliente = spark.sql(f'SELECT IdCliente, COUNT(*) AS qtd FROM {T_TRANSACOES} GROUP BY IdCliente').toPandas()
univariada(tx_cliente['qtd'], 'Transações por cliente (vida toda)', 'T3_transacoes_por_cliente',
           xlabel='nº de transações do cliente', bins=50)

### 2.2 Transações × Transação produto
Quantos itens cada transação tem e se os pontos batem com a soma dos itens.

In [0]:
spark.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {T_TRANSACOES})                                          AS transacoes,
        (SELECT COUNT(DISTINCT IdTransacao) FROM {T_TRANSACAO_PRODUTO})                AS transacoes_com_produto,
        (SELECT COUNT(*) FROM {T_TRANSACOES} t
            LEFT JOIN (SELECT DISTINCT IdTransacao FROM {T_TRANSACAO_PRODUTO}) tp
            ON t.IdTransacao = tp.IdTransacao WHERE tp.IdTransacao IS NULL)            AS transacoes_sem_produto,
        (SELECT COUNT(*) FROM {T_TRANSACAO_PRODUTO} tp
            LEFT JOIN {T_TRANSACOES} t ON tp.IdTransacao = t.IdTransacao
            WHERE t.IdTransacao IS NULL)                                               AS itens_sem_transacao
""").toPandas().T.rename(columns={0: 'qtd'})

In [0]:
# quantos produtos cada transação tem? se for sempre 1, o join não duplica linhas
itens = spark.sql(f"""
    SELECT qtd_itens, COUNT(*) AS transacoes
    FROM (SELECT IdTransacao, COUNT(*) AS qtd_itens FROM {T_TRANSACAO_PRODUTO} GROUP BY IdTransacao)
    GROUP BY qtd_itens ORDER BY qtd_itens
""").toPandas()
itens['%'] = (itens['transacoes'] / itens['transacoes'].sum() * 100).round(2)
itens

In [0]:
# consistência: os pontos da transação batem com a soma dos itens (vlProduto x QtdeProduto)?
spark.sql(f"""
    WITH itens AS (
        SELECT IdTransacao, SUM(vlProduto * QtdeProduto) AS pontos_itens
        FROM {T_TRANSACAO_PRODUTO} GROUP BY IdTransacao
    )
    SELECT COUNT(*)                                                        AS transacoes_comparadas,
           ROUND(AVG(CASE WHEN t.QtdePontos = i.pontos_itens THEN 1 ELSE 0 END) * 100, 2) AS pct_batem
    FROM {T_TRANSACOES} t JOIN itens i ON t.IdTransacao = i.IdTransacao
""").toPandas()

### 2.3 Transação produto × Produtos
Itens sem produto no catálogo e quais produtos concentram o uso. 

In [0]:
spark.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {T_TRANSACAO_PRODUTO} tp
            LEFT JOIN {T_PRODUTOS} p ON tp.IdProduto = p.IdProduto
            WHERE p.IdProduto IS NULL)                                                 AS itens_com_produto_sem_cadastro,
        (SELECT COUNT(*) FROM {T_PRODUTOS} p
            LEFT JOIN (SELECT DISTINCT IdProduto FROM {T_TRANSACAO_PRODUTO}) tp
            ON p.IdProduto = tp.IdProduto WHERE tp.IdProduto IS NULL)                  AS produtos_nunca_usados
""").toPandas().T.rename(columns={0: 'qtd'})

In [0]:
# uso de cada produto na vida toda (antes de filtrar por safra)
uso = spark.sql(f"""
    SELECT p.DescCategoriaProduto AS categoria, p.DescNomeProduto AS produto,
           COUNT(*) AS itens, SUM(tp.vlProduto * tp.QtdeProduto) AS pontos
    FROM {T_TRANSACAO_PRODUTO} tp
    JOIN {T_PRODUTOS} p ON tp.IdProduto = p.IdProduto
    GROUP BY 1, 2 ORDER BY itens DESC
""").toPandas()
uso['% itens'] = (uso['itens'] / uso['itens'].sum() * 100).round(2)
uso

### 2.4 Saldo do cadastro × soma das transações
Confirma se `qtdePontos` (clientes) é uma foto atual do saldo, e por isso não pode virar feature.

In [0]:
# o saldo do cadastro bate com a soma de pontos das transações? (ajuda a confirmar que qtdePontos é foto atual)
spark.sql(f"""
    WITH saldo AS (SELECT IdCliente, SUM(QtdePontos) AS soma_transacoes FROM {T_TRANSACOES} GROUP BY IdCliente)
    SELECT COUNT(*) AS clientes_comparados,
           ROUND(AVG(CASE WHEN c.qtdePontos = s.soma_transacoes THEN 1 ELSE 0 END) * 100, 2) AS pct_batem
    FROM {T_CLIENTES} c JOIN saldo s ON c.idCliente = s.IdCliente
""").toPandas()

## 3. Conclusões



**Dados:** 4 tabelas com chaves únicas e sem duplicatas. Transações de 27/01/2024 a 07/09/2026 (jan/24 e set/26 incompletos).

**Achados**
- **Base ativa encolhendo:** de 675 clientes/mês (fev/24) para ~250 (2026).
- **35% dos clientes nunca transacionaram** (2.010 de 5.785).
- **Atividade concentrada:** mediana de 8 transações por cliente, máximo de 31.244 → investigar possíveis contas atípicas.
- **Engajamento ≫ loja:** 99,2% das transações são de ganho e só 0,77% de gasto.
- **5 produtos = 98% dos itens:** ChatMessage (82%), Lista de presença (14%), Presença Streak, Resgatar Ponei e Troca StreamElements.

**Qualidade**
- `flBlueSky` e `flInstagram` são constantes → descartar.
- `qtdePontos` e `DtAtualizacao` (clientes) são foto atual → não usar como feature (vazamento).
- 0,19% dos itens sem produto no catálogo (158 nulos + 514 ids não encontrados).
- Pontos da transação batem com os itens em 99,93% → fonte de verdade: `transacoes.QtdePontos`.

**Joins**
- `clientes 1:N transacoes` (`idCliente = IdCliente`)
- `transacoes 1:1 transacao_produto` em 99,94% (`IdTransacao`) → agregar antes de somar
- `transacao_produto N:1 produtos` (`IdProduto`) → `LEFT JOIN`